## Manifests, digests & multi-arch

When you push, the registry stores **blobs** (layers, config) and **manifests** (JSON pointing at those blobs). The manifest is the source of truth for "what is this image." A simplified single-platform manifest:

```json
{
  "schemaVersion": 2,
  "config": { "digest": "sha256:0123...", "size": 7023 },
  "layers": [
    { "digest": "sha256:abcd...", "size": 31405122 },
    { "digest": "sha256:efgh...", "size": 2451 }
  ]
}
```

That JSON, **exactly as serialised, is the canonical bytes of the image.** SHA-256 those bytes and you get the image's **digest**. Change anything — a layer order, one byte of config — and the digest changes. That's *why* digest = identity: it can't be tampered with without becoming a different image. Inspect it with `docker buildx imagetools inspect <ref>` (add `--raw` for the actual JSON).

**Multi-arch — one tag, many CPUs.** A single tag like `nginx:alpine` runs on an Intel Mac, Apple Silicon, an x86 server, and an ARM Pi. How? The tag points not at one manifest but at a **manifest list** (an OCI Image Index) — a thin JSON object listing a per-architecture manifest:

```
nginx:alpine ──► manifest list ──┬─► linux/amd64 → manifest A → layers
                                 ├─► linux/arm64 → manifest B → layers
                                 └─► linux/arm/v7 → manifest C → layers
```

`docker pull` on an ARM Mac fetches the list, picks the `linux/arm64` entry, and pulls **only that** manifest's layers — the other architectures never download. You build one yourself with buildx across platforms:

```bash
docker buildx build --platform linux/amd64,linux/arm64 -t ghcr.io/myorg/api:1.2.3 --push .
```

`--push` is mandatory — a manifest list can only live in a registry, not a local image store. After that one command, the tag works on both architectures.